In [ ]:
///- --package ..
:< sm'_operators
:< engine

#### site (test)

In [ ]:
///- --test static

:< testing

In [1]:
:== file = { path : string; content : string }

:= layout (title : string) (body : string) : string =
    "<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
    ++# "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">"
    ++# "<title>" ++# html.escape title ++# "</title>"
    ++# "<link rel=\"stylesheet\" href=\"/style.css\"></head>"
    ++# "<body><header><a class=\"home\" href=\"/\">Spiral Blog</a></header><main>"
    ++# body ++# "</main></body></html>"

:= entry_file (e : page.entry) : file =
    { path = e.slug ++# "/index.html"; content = layout e.title (page.render_entry e) }

:= ^ entry_files (entries : list page.entry) : list file =
    ?= entries ?|
    | [] => []
    | e :: rest => entry_file e :: entry_files rest

:= base_url () : string = "https://i574n.github.io/blog"

:= files (entries : list page.entry) : list file =
    := index_page = { path = "index.html"; content = layout "Spiral Blog" (index.render_index entries) }
    := stylesheet = { path = "style.css"; content = style.css () }
    := feed = { path = "rss.xml"; content = rss.render_feed "Spiral Blog" (base_url ()) entries }
    := articles_page =
        {
            path = "articles/index.html"
            content = layout "Articles - Spiral Blog" (index.render_index (index.only page.Article entries))
        }
    := releases_page =
        {
            path = "releases/index.html"
            content = layout "Releases - Spiral Blog" (index.render_index (index.only page.Release entries))
        }
    index_page :: stylesheet :: feed :: articles_page :: releases_page :: entry_files entries

:= ^ paths (fs : list file) : string =
    ?= fs ?|
    | [] => ""
    | [ f ] => f.path
    | f :: rest => f.path ++# " " ++# paths rest

#### the_site_has_an_index_and_one_folder_per_post (test)

In [2]:
///- --test

files [ page.sample () ]
|> paths
|> _assert_eq "index.html style.css rss.xml articles/index.html releases/index.html sample/index.html"

__assert_eq { expected = index.html style.css rss.xml articles/index.html releases/index.html sample/index.html }

#### every_page_shares_the_layout (test)

In [3]:
///- --test

:= ^ all_wrapped (fs : list file) : bool =
    ?= fs ?|
    | [] => !+
    | f :: rest ?& f.path |> sm'.ends_with ".html" =>
        (f.content |> sm'.starts_with "<!doctype html>")
        && (f.content |> sm'.ends_with "</main></body></html>")
        && all_wrapped rest
    | _ :: rest => all_wrapped rest
files [ page.sample () ] |> all_wrapped |> _assert_eq !+

__assert_eq { expected = true }

#### the_post_page_title_is_escaped_into_the_head (test)

In [4]:
///- --test

:= e = page.sample ()
files [ { e ?| title = "A & B" } ]
|> \ fs =>
    ?= fs ?|
    | _ :: _ :: _ :: _ :: _ :: post_file :: _ =>
        post_file.content |> sm'.contains "<title>A &amp; B</title>" |> _assert_eq !+
    | _ => failwith "expected a post file"

__assert_eq { expected = true }